In [24]:
from pathlib import Path
import json
import time
import pandas as pd

print("Project folder:", Path.cwd())
print("Matcher exists:", Path("job_matcher.py").exists())
print("Results file exists:", Path("job_match_results.json").exists())
print("Resume folder exists:", Path("resumes").exists())

Project folder: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching
Matcher exists: True
Results file exists: True
Resume folder exists: True


In [25]:

import job_matcher as jm

resume_folder = Path("resumes")
resume_files = [
    p for p in resume_folder.iterdir()
    if p.is_file() and p.suffix.lower() in {".pdf", ".docx", ".txt"}
]

print("Resume files:", len(resume_files))
print("Resume chunks in ChromaDB:", jm.collection.count())

all_metadata = jm.collection.get(include=["metadatas"])
unique_ids = {
    metadata["resume_id"]
    for metadata in all_metadata["metadatas"]
    if metadata and metadata.get("resume_id")
}

print("Unique indexed resumes:", len(unique_ids))


Resume files: 14
Resume chunks in ChromaDB: 76
Unique indexed resumes: 13


In [26]:

start_time = time.perf_counter()

jm.match_candidates()

elapsed_seconds = time.perf_counter() - start_time

print(f"End-to-end latency: {elapsed_seconds:.2f} seconds")



Available Job Descriptions:
1. data_engineer_JD
2. devops_JD
3. frontend_react_JD
4. java_springboot_JD
5. python_ml_JD



Select a job description (1-5):  4



Selected: java_springboot_JD.txt

Extracting job skills...
Job skills:  ['sql', 'java', 'git']
Must have skills:  None detected

Generating job description embedding...
Searching the resume database...

Top eligible candidates

1. JOHN DOE
   Match score: 86.61/100
   Eligibility: Eligible
   Matched skills: sql, java, git
   Explanation: Matching evidence: sql, java, git.Skills not found in retrieved resume text: None of the listed job skills are missing.Check the original resume before making a decision.
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\resume_john_doe.pdf

2. Jake Ryan
   Match score: 77.06/100
   Eligibility: Eligible
   Matched skills: sql, java, git
   Explanation: Matching evidence: sql, java, git.Skills not found in retrieved resume text: None of the listed job skills are missing.Check the original resume before making a decision.
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\Jake_Ryan_Resum

In [31]:

with open("job_match_results.json", encoding="utf-8") as f:
    results = json.load(f)

candidates = results["top_matches"]

results_table = pd.DataFrame([
    {
        "Rank": i,
        "Candidate": c["candidate_name"],
        "Score": c["match_score"],
        "Matched skills": ", ".join(c["matched_skills"]),
        "Eligibility": c["eligibility"]
    }
    for i, c in enumerate(candidates, start=1)
])

display(results_table)


,Rank,Candidate,Score,Matched skills,Eligibility
0,1,JOHN DOE,86.61,"sql, java, git",Eligible
1,2,Jake Ryan,77.06,"sql, java, git",Eligible
2,3,David_Cole_Resume.pdf,75.84,"sql, java, git",Eligible
3,4,Viraj_Soni_Resume.pdf,59.03,"java, git",Eligible
4,5,Deep_Mehta_Resume.pdf,56.31,"sql, java",Eligible
5,6,ALEX MEHTA,55.61,"sql, git",Eligible
6,7,EMILY SHARMA,55.15,"sql, git",Eligible
7,8,Alok Pandey,44.81,java,Eligible
8,9,Neha Kapoor,43.77,java,Eligible
9,10,KARAN PRATAP SINGH,36.98,java,Eligible


In [33]:

# Automatically select candidates scoring above 70%
relevant_names = {
    candidate["candidate_name"].strip().casefold()
    for candidate in candidates
    if float(candidate["match_score"]) > 70
}

print("Relevant candidates (score > 70%):")
for candidate in candidates:
    if candidate["candidate_name"].strip().casefold() in relevant_names:
        print(
            candidate["candidate_name"],
            "-", candidate["match_score"], "%"
        )

print("Total relevant candidates:", len(relevant_names))



Relevant candidates (score > 70%):
JOHN DOE - 86.61 %
Jake Ryan - 77.06 %
David_Cole_Resume.pdf - 75.84 %
Total relevant candidates: 3


In [35]:

# Normalize names so capitalization and extra spaces don't matter
relevant_names_normalized = {
    name.strip().casefold()
    for name in relevant_names
}

top_10 = candidates[:10]

relevant_in_top_10 = sum(
    c["candidate_name"].strip().casefold() in relevant_names_normalized
    for c in top_10
)

precision_at_10 = (
    relevant_in_top_10 / len(top_10)
    if top_10 else 0.0
)

recall_at_10 = (
    relevant_in_top_10 / len(relevant_names_normalized)
    if relevant_names_normalized else 0.0
)

print(f"Relevant candidates in top 10: {relevant_in_top_10}")
print(f"Candidates returned: {len(top_10)}")
print(f"Precision@10: {precision_at_10:.2f}")
print(f"Recall@10: {recall_at_10:.2f}")
print(f"End-to-end latency: {elapsed_seconds:.2f} seconds")


Relevant candidates in top 10: 3
Candidates returned: 10
Precision@10: 0.30
Recall@10: 1.00
End-to-end latency: 5.79 seconds


In [36]:
for i, candidate in enumerate(candidates, start=1):
    print(f"{i}. {candidate['candidate_name']}")
    print("   Score:", candidate["match_score"])
    print("   Matched skills:", candidate["matched_skills"])
    print("   Resume:", candidate["resume_path"])
    print()

1. JOHN DOE
   Score: 86.61
   Matched skills: ['sql', 'java', 'git']
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\resume_john_doe.pdf

2. Jake Ryan
   Score: 77.06
   Matched skills: ['sql', 'java', 'git']
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\Jake_Ryan_Resume.pdf.pdf

3. David_Cole_Resume.pdf
   Score: 75.84
   Matched skills: ['sql', 'java', 'git']
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\David_Cole_Resume.pdf.pdf

4. Viraj_Soni_Resume.pdf
   Score: 59.03
   Matched skills: ['java', 'git']
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\Viraj_Soni_Resume.pdf.pdf

5. Deep_Mehta_Resume.pdf
   Score: 56.31
   Matched skills: ['sql', 'java']
   Resume: C:\Users\UDITA BAGGA\python-practice\rag_based_profile_matching\resumes\Deep_Mehta_Resume.pdf.pdf

6. ALEX MEHTA
   Score: 55.61
   Matched skills: ['sql', 'git']
   Resume: C:\Use

In [37]:
import json

with open("job_match_results.json", encoding="utf-8") as f:
    results = json.load(f)

print("JOB DESCRIPTION:")
print(results.get("job_description", "No job description saved"))

print("\nSKILLS DETECTED:")
print(results.get("job_skills", []))

JOB DESCRIPTION:
We are looking for a Java Backend Developer with knowledge of Java, Spring Boot, and object-oriented programming. 
Candidates should have experience with REST APIs, microservices, and SQL databases. 
Familiarity with Hibernate, Git, and Maven is preferred.

SKILLS DETECTED:
['sql', 'java', 'git']


In [38]:

import job_matcher as jm

job_text = results["job_description"]

print("Test 1 — Python detected:")
print(jm.contains_skill(job_text, "python"))

print("\nTest 2 — SQL detected:")
print(jm.contains_skill(job_text, "sql"))

print("\nTest 3 — All detected job skills:")
print(jm.extract_job_skills(job_text))

print("\nTest 4 — Skills configured in the matcher:")
print(jm.SKILL_TERMS)


Test 1 — Python detected:
False

Test 2 — SQL detected:
True

Test 3 — All detected job skills:
['sql', 'java', 'git']

Test 4 — Skills configured in the matcher:
['python', 'sql', 'machine learning', 'deep learning', 'artificial intelligence', 'data analysis', 'data science', 'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'pytorch', 'natural language processing', 'nlp', 'computer vision', 'java', 'javascript', 'typescript', 'c++', 'c#', 'html', 'css', 'react', 'node.js', 'django', 'flask', 'fastapi', 'rest api', 'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'git', 'linux', 'excel', 'power bi', 'tableau', 'statistics', 'postgresql', 'mysql', 'mongodb', 'spark', 'hadoop', 'etl', 'data engineering', 'llm', 'generative ai', 'rag', 'langchain']


In [39]:
import importlib
import job_matcher as jm

importlib.reload(jm)

job_text = results["job_description"]

print("Python:", jm.contains_skill(job_text, "python"))
print("SQL:", jm.contains_skill(job_text, "sql"))
print("Detected skills:", jm.extract_job_skills(job_text))

Python: False
SQL: True
Detected skills: ['sql', 'java', 'git']


In [40]:
with open("job_match_results.json", encoding="utf-8") as f:
    results = json.load(f)

candidates = results["top_matches"]

for c in candidates:
    print(c["candidate_name"], c["matched_skills"])

JOHN DOE ['sql', 'java', 'git']
Jake Ryan ['sql', 'java', 'git']
David_Cole_Resume.pdf ['sql', 'java', 'git']
Viraj_Soni_Resume.pdf ['java', 'git']
Deep_Mehta_Resume.pdf ['sql', 'java']
ALEX MEHTA ['sql', 'git']
EMILY SHARMA ['sql', 'git']
Alok Pandey ['java']
Neha Kapoor ['java']
KARAN PRATAP SINGH ['java']


In [41]:
with open("job_match_results.json", encoding="utf-8") as f:
    results = json.load(f)

candidates = results["top_matches"]

print("Unique candidates retrieved:",
      results["total_unique_candidates_retrieved"])

for i, c in enumerate(candidates, start=1):
    print(f"\n{i}. {c['candidate_name']}")
    print("   Score:", c["match_score"])
    print("   Matched skills:", c["matched_skills"])
    print("   Missing skills:", c["missing_skills"])

Unique candidates retrieved: 13

1. JOHN DOE
   Score: 86.61
   Matched skills: ['sql', 'java', 'git']
   Missing skills: []

2. Jake Ryan
   Score: 77.06
   Matched skills: ['sql', 'java', 'git']
   Missing skills: []

3. David_Cole_Resume.pdf
   Score: 75.84
   Matched skills: ['sql', 'java', 'git']
   Missing skills: []

4. Viraj_Soni_Resume.pdf
   Score: 59.03
   Matched skills: ['java', 'git']
   Missing skills: ['sql']

5. Deep_Mehta_Resume.pdf
   Score: 56.31
   Matched skills: ['sql', 'java']
   Missing skills: ['git']

6. ALEX MEHTA
   Score: 55.61
   Matched skills: ['sql', 'git']
   Missing skills: ['java']

7. EMILY SHARMA
   Score: 55.15
   Matched skills: ['sql', 'git']
   Missing skills: ['java']

8. Alok Pandey
   Score: 44.81
   Matched skills: ['java']
   Missing skills: ['sql', 'git']

9. Neha Kapoor
   Score: 43.77
   Matched skills: ['java']
   Missing skills: ['sql', 'git']

10. KARAN PRATAP SINGH
   Score: 36.98
   Matched skills: ['java']
   Missing skills: ['sq

In [41]:
print("Your relevance labels:")
print(relevant_names)

print("\nActual candidate names:")
for c in candidates:
    print(repr(c["candidate_name"]))

Your relevance labels:
{'ALEX MEHTA', 'Jake Ryan', 'EMILY SHARMA'}

Actual candidate names:
'ALEX MEHTA'
'Jake Ryan'
'EMILY SHARMA'
'resume'


In [42]:
actual_names = {
    c["candidate_name"] for c in candidates
}

print("Labels matching returned candidates:")
print(actual_names.intersection(relevant_names))

print("Number of relevant names entered:", len(relevant_names))

Labels matching returned candidates:
set()
Number of relevant names entered: 3
